In [ ]:
# Hücre 0 — Kurulum
# Colab Pro'da ilk çalıştırmadan önce bir kez çalıştır.
!pip install pennylane pennylane-lightning kaggle -q
import os
os.makedirs('/content/training/data', exist_ok=True)
os.makedirs('/content/training/artifacts', exist_ok=True)
print('✓ Paketler hazır')

In [ ]:
# Hücre 1 — Drive bağla + import + sabitler
# Bu hücreyi her session başında çalıştır.
from google.colab import userdata, drive
from pathlib import Path
import os, glob, time
import numpy as np
import tensorflow as tf
import pennylane as qml
import pennylane.numpy as pnp

drive.mount('/content/drive')

# ── Mimari sabitleri ──────────────────────────────────────────────────────────
IMG_SIZE     = 128
FEATURE_DIM  = 128   # MobileNetV2 Dense çıktı boyutu
N_QUBITS     = 7     # 2^7 = 128 = FEATURE_DIM (AmplitudeEmbedding için tam uyum)
N_LAYERS     = 4     # StronglyEntanglingLayers derinliği
OUTPUT_DIM   = N_QUBITS * 3   # 21 (PauliX + PauliY + PauliZ, her qubit)
BATCH_SIZE   = 32
BATCH_SIZE_Q = 128
EPOCHS_P1    = 15
EPOCHS_P2    = 25
UNFREEZE_N   = 30    # MobileNetV2'nin son kaç katmanı açık eğitilsin

# ── Drive klasörleri ──────────────────────────────────────────────────────────
PI_MODEL_DIR = Path('/content/drive/MyDrive/plant_disease_pi/models')
PI_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Checkpoint yolları ────────────────────────────────────────────────────────
PHASE1_CKPT  = PI_MODEL_DIR / 'phase1_best_weights.weights.h5'
TFLITE_PATH  = PI_MODEL_DIR / 'feature_extractor.tflite'
FEAT_TR      = PI_MODEL_DIR / 'X128_tr.npy'
FEAT_VAL     = PI_MODEL_DIR / 'X128_val.npy'
LABEL_TR     = PI_MODEL_DIR / 'y_tr.npy'
LABEL_VAL    = PI_MODEL_DIR / 'y_val.npy'
PHASE2_STATE = PI_MODEL_DIR / 'phase2_state.npz'
BEST_WEIGHTS = PI_MODEL_DIR / 'quantum_weights.npz'

print('GPU        :', tf.config.list_physical_devices('GPU'))
print('PennyLane  :', qml.__version__)
print('Drive klas.:', PI_MODEL_DIR)

In [ ]:
# Hücre 1b — TFLite Onarım  ← SADECE BUNU ÇALIŞTIR (dataset gerekmez)
# Mevcut Phase 1 checkpoint'ten 128D çıktılı TFLite'ı yeniden oluşturur.
# Hücre 0 ve 1'den sonra çalıştır, sonra Drive'dan feature_extractor.tflite indir.

if not PHASE1_CKPT.exists():
    raise FileNotFoundError(f'Phase 1 checkpoint bulunamadı: {PHASE1_CKPT}')

# class_names.txt'ten sınıf sayısını oku
n_cls = len((PI_MODEL_DIR / 'class_names.txt').read_text(encoding='utf-8').strip().split('\n'))
print(f'Sınıf sayısı: {n_cls}, Feature dim: {FEATURE_DIM}')

# Modeli yeniden inşa et (checkpoint yüklemek için tam yapı gerekir)
base_r = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None
)
inp_r  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x_r    = tf.keras.applications.mobilenet_v2.preprocess_input(inp_r)
x_r    = base_r(x_r)
x_r    = tf.keras.layers.GlobalAveragePooling2D()(x_r)
x_r    = tf.keras.layers.Dropout(0.2)(x_r)
feat_r = tf.keras.layers.Dense(FEATURE_DIM, activation='relu', name='features')(x_r)
out_r  = tf.keras.layers.Dense(n_cls, activation='softmax', name='head')(feat_r)

repair_model = tf.keras.Model(inp_r, out_r)
repair_model.load_weights(str(PHASE1_CKPT))
print('✓ Checkpoint yüklendi')

# Sadece feature extractor kısmını TFLite'a çevir
feat_model_r = tf.keras.Model(inp_r, feat_r, name='feat_model')

# Çıktı boyutunu doğrula
dummy = tf.zeros((1, IMG_SIZE, IMG_SIZE, 3))
out_shape = feat_model_r(dummy).shape
print(f'✓ Feature extractor çıktı boyutu: {out_shape}  (128 olmalı)')

converter    = tf.lite.TFLiteConverter.from_keras_model(feat_model_r)
tflite_bytes = converter.convert()
TFLITE_PATH.write_bytes(tflite_bytes)
print(f'✓ feature_extractor.tflite güncellendi ({len(tflite_bytes)//1024} KB)')
print(f'  Drive konumu: {TFLITE_PATH}')
print()
print('Şimdi bu dosyayı Drive\'dan indirip models/ klasörüne koy.')

In [ ]:
# Hücre 2 — Dataset indir
# Kaggle kimlik bilgilerini Colab Secrets'a ekle:
#   Sol panel → 🔑 simgesi → KAGGLE_USERNAME ve KAGGLE_KEY ekle

LOCAL_DATA = Path('/content/training/data')
train_paths = glob.glob(str(LOCAL_DATA / '**/train'), recursive=True)

if not train_paths:
    print('Dataset yok, Kaggle\'dan indiriliyor...')
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    os.chdir(str(LOCAL_DATA))
    os.system('kaggle datasets download -d vipoooool/new-plant-diseases-dataset --unzip -q')
    train_paths = glob.glob(str(LOCAL_DATA / '**/train'), recursive=True)
    print('✓ İndirildi')
else:
    print('✓ Dataset zaten mevcut')

TRAIN_DIR = Path(sorted(train_paths)[0])
VALID_DIR = Path(sorted(glob.glob(str(LOCAL_DATA / '**/valid'), recursive=True))[0])
print('Train:', TRAIN_DIR)
print('Valid:', VALID_DIR)

In [ ]:
# Hücre 3 — Dataset yükle + augmentation pipeline
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='int', shuffle=True
)
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='int', shuffle=False
)

class_names = train_ds_raw.class_names
N_CLASSES   = len(class_names)
print(f'{N_CLASSES} sınıf')

(PI_MODEL_DIR / 'class_names.txt').write_text('\n'.join(class_names), encoding='utf-8')
print('✓ class_names.txt kaydedildi')

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name='augmentation')

def apply_aug(images, labels):
    return augment(images, training=True), labels

train_ds = (
    train_ds_raw
    .map(apply_aug, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = val_ds_raw.prefetch(tf.data.AUTOTUNE)
print('✓ Pipeline hazır')

In [ ]:
# Hücre 4 — Phase 1: MobileNetV2 fine-tune

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
)
for layer in base.layers[:-UNFREEZE_N]:
    layer.trainable = False
for layer in base.layers[-UNFREEZE_N:]:
    layer.trainable = True

inp  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x    = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
x    = base(x, training=False)
x    = tf.keras.layers.GlobalAveragePooling2D()(x)
x    = tf.keras.layers.Dropout(0.2)(x)
feat = tf.keras.layers.Dense(FEATURE_DIM, activation='relu', name='features')(x)
out  = tf.keras.layers.Dense(N_CLASSES,   activation='softmax', name='head')(feat)

feat_model       = tf.keras.Model(inp, feat, name='feat_model')
classifier_model = tf.keras.Model(inp, out,  name='classifier')
classifier_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

_phase1_retrained = False

if PHASE1_CKPT.exists():
    classifier_model.load_weights(str(PHASE1_CKPT))
    val_acc = classifier_model.evaluate(val_ds, verbose=0)[1]
    print(f'✓ Phase 1 checkpoint yüklendi — val_accuracy={val_acc:.4f}')
else:
    _phase1_retrained = True
    print('Phase 1 başlıyor...')
    history = classifier_model.fit(
        train_ds, epochs=EPOCHS_P1, validation_data=val_ds,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                str(PHASE1_CKPT), save_best_only=True,
                save_weights_only=True, monitor='val_accuracy', verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1),
            tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, verbose=1),
        ]
    )
    print(f'✓ Phase 1 val_accuracy: {max(history.history["val_accuracy"]):.4f}')

# TFLite her zaman backbone ile tutarlı olacak şekilde koşulsuz export edilir.
# "Zaten mevcut" kontrolü YOK — backbone değişirse TFLite de değişmeli.
converter    = tf.lite.TFLiteConverter.from_keras_model(feat_model)
tflite_bytes = converter.convert()
TFLITE_PATH.write_bytes(tflite_bytes)
print(f'✓ feature_extractor.tflite ({len(tflite_bytes)//1024} KB) güncellendi')

In [ ]:
# Hücre 5 — Feature extraction + L2 normalize
# Phase 1 yeniden eğitildiyse cache ve quantum checkpoint otomatik temizlenir.

if _phase1_retrained:
    print('⚠ Phase 1 yeniden eğitildi → cache ve quantum checkpoint temizleniyor...')
    for f in [FEAT_TR, FEAT_VAL, LABEL_TR, LABEL_VAL, PHASE2_STATE]:
        if f.exists():
            f.unlink()

if FEAT_TR.exists() and FEAT_VAL.exists():
    print('Drive cache bulundu, yükleniyor...')
    X128_tr  = np.load(str(FEAT_TR))
    X128_val = np.load(str(FEAT_VAL))
    y_tr     = np.load(str(LABEL_TR))
    y_val    = np.load(str(LABEL_VAL))
else:
    print('Feature extraction (augmentasyonsuz)...')
    def extract(ds, model):
        X, y = [], []
        for imgs, labels in ds:
            X.append(model(imgs, training=False).numpy())
            y.extend(labels.numpy())
        return np.vstack(X), np.array(y)

    clean_train = train_ds_raw.prefetch(tf.data.AUTOTUNE)
    clean_val   = val_ds_raw.prefetch(tf.data.AUTOTUNE)
    X128_tr,  y_tr  = extract(clean_train, feat_model)
    X128_val, y_val = extract(clean_val,   feat_model)

    np.save(str(FEAT_TR),   X128_tr)
    np.save(str(FEAT_VAL),  X128_val)
    np.save(str(LABEL_TR),  y_tr)
    np.save(str(LABEL_VAL), y_val)
    print('✓ Drive cache kaydedildi')

print(f'Train: {X128_tr.shape}, Val: {X128_val.shape}')

def l2_norm(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)

X_tr_norm  = l2_norm(X128_tr).astype(np.float64)
X_val_norm = l2_norm(X128_val).astype(np.float64)
y_tr_oh    = np.eye(N_CLASSES)[y_tr]
print('✓ L2 normalize tamamlandı')

In [ ]:
# Hücre 6 — Phase 2: Quantum eğitim
#
# Mimari:
#   128D özellik (L2 norm) → AmplitudeEmbedding (7 qubit, 2^7=128 amplitüd)
#   → StronglyEntanglingLayers (N_LAYERS=4) → 21D ölçüm
#   → Lineer sınıflandırıcı → 38 sınıf
#
# Her epoch sonunda phase2_state.npz Drive'a yazılır.
# Notebook kapanırsa aynı hücreyi çalıştır; kaldığı epoch'tan devam eder.

try:
    dev_q = qml.device('lightning.qubit', wires=N_QUBITS)
    DIFF_METHOD = 'adjoint'
    print('✓ lightning.qubit (adjoint diff)')
except Exception as e:
    dev_q = qml.device('default.qubit', wires=N_QUBITS)
    DIFF_METHOD = 'best'
    print(f'⚠ lightning.qubit yüklenemedi ({e}), default.qubit kullanılıyor')

@qml.qnode(dev_q, interface='autograd', diff_method=DIFF_METHOD)
def q_circ(features, weights):
    qml.AmplitudeEmbedding(features, wires=range(N_QUBITS), normalize=False)
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return (
        [qml.expval(qml.PauliX(i)) for i in range(N_QUBITS)] +
        [qml.expval(qml.PauliY(i)) for i in range(N_QUBITS)] +
        [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    )

start_epoch = 1
best_acc    = 0.0

if PHASE2_STATE.exists():
    state       = np.load(str(PHASE2_STATE))
    q_w         = pnp.array(state['q_w'], requires_grad=True)
    C_W         = pnp.array(state['C_W'], requires_grad=True)
    C_b         = pnp.array(state['C_b'], requires_grad=True)
    start_epoch = int(state['last_epoch']) + 1
    best_acc    = float(state['best_acc'])
    print(f'✓ Phase 2 checkpoint: epoch {start_epoch} devam ediyor (best={best_acc:.4f})')
else:
    q_w = pnp.array(
        np.random.uniform(-np.pi/4, np.pi/4, (N_LAYERS, N_QUBITS, 3)),
        requires_grad=True
    )
    C_W = pnp.array(np.random.randn(OUTPUT_DIM, N_CLASSES) * 0.01, requires_grad=True)
    C_b = pnp.zeros(N_CLASSES, requires_grad=True)
    print(f'Phase 2 sıfırdan: epochs={EPOCHS_P2}, qubits={N_QUBITS}, '
          f'layers={N_LAYERS}, AmplitudeEmbedding')

def cost(qw, cw, cb, Xb, yb):
    q_out = pnp.stack([pnp.array(q_circ(Xb[i], qw)) for i in range(len(Xb))])
    e     = pnp.exp(q_out @ cw + cb)
    probs = e / pnp.sum(e, axis=1, keepdims=True)
    return -pnp.mean(pnp.sum(yb * pnp.log(probs + 1e-9), axis=1))

def cosine_lr(epoch, total=EPOCHS_P2, lr_max=0.01, lr_min=1e-4, T0=12):
    cycle = (epoch - 1) % T0
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * cycle / T0))

opt = qml.AdamOptimizer(stepsize=0.01)

for epoch in range(start_epoch, EPOCHS_P2 + 1):
    t0 = time.time()
    opt.stepsize = cosine_lr(epoch)

    idx    = np.random.permutation(len(X_tr_norm))
    X_sh   = X_tr_norm[idx]
    y_sh   = y_tr_oh[idx]
    n_bat  = len(X_sh) // BATCH_SIZE_Q
    losses = []

    for i in range(0, n_bat * BATCH_SIZE_Q, BATCH_SIZE_Q):
        Xb = X_sh[i:i + BATCH_SIZE_Q]
        yb = y_sh[i:i + BATCH_SIZE_Q]
        (q_w, C_W, C_b), loss = opt.step_and_cost(cost, q_w, C_W, C_b, Xb=Xb, yb=yb)
        losses.append(float(loss))
        if (i // BATCH_SIZE_Q) % 50 == 0:
            print(f'  epoch {epoch}  batch {i//BATCH_SIZE_Q}/{n_bat}'
                  f'  loss={loss:.4f}  lr={opt.stepsize:.5f}')

    val_logits = []
    for j in range(0, len(X_val_norm), BATCH_SIZE_Q):
        Xb_v   = X_val_norm[j:j + BATCH_SIZE_Q]
        q_out  = np.stack([np.array(q_circ(Xb_v[k], np.array(q_w)))
                           for k in range(len(Xb_v))])
        val_logits.append(q_out @ np.array(C_W) + np.array(C_b))
    logits = np.vstack(val_logits)
    acc    = float(np.mean(np.argmax(logits, axis=1) == y_val))

    elapsed = time.time() - t0
    print(f'★ Epoch {epoch}/{EPOCHS_P2}  val_acc={acc:.4f}  '
          f'loss={np.mean(losses):.4f}  lr={opt.stepsize:.5f}  '
          f't={elapsed:.0f}s', end='')

    state_dict = dict(q_w=np.array(q_w), C_W=np.array(C_W), C_b=np.array(C_b),
                      last_epoch=np.array(epoch), best_acc=np.array(max(acc, best_acc)))
    np.savez(str(PHASE2_STATE), **state_dict)

    if acc > best_acc:
        best_acc = acc
        np.savez(
            str(BEST_WEIGHTS),
            quantum_weights   = np.array(q_w),
            classifier_kernel = np.array(C_W),
            classifier_bias   = np.array(C_b),
            n_layers          = np.array(N_LAYERS),
            n_qubits          = np.array(N_QUBITS),
            feature_dim       = np.array(FEATURE_DIM),
        )
        print('  ← EN İYİ kaydedildi ✓')
    else:
        print(f'  (best={best_acc:.4f})')

print(f'\n✓ Bitti. En iyi val_accuracy: {best_acc:.4f}')
print('Drive klasöründe bulunan dosyalar:')
for f in sorted(PI_MODEL_DIR.iterdir()):
    print(f'  {f.name:40s} {f.stat().st_size//1024:6d} KB')

In [ ]:
# Hücre 7 — Pi klasörüne kopyala + özet
import shutil

LOCAL_PI = Path('/content/plant_disease_pi/models')
LOCAL_PI.mkdir(parents=True, exist_ok=True)

for fname in ['feature_extractor.tflite', 'quantum_weights.npz', 'class_names.txt']:
    src = PI_MODEL_DIR / fname
    if src.exists():
        shutil.copy(str(src), str(LOCAL_PI / fname))
        print(f'✓ {fname:40s} {src.stat().st_size//1024:6d} KB')
    else:
        print(f'⚠ {fname} bulunamadı!')

data = np.load(str(BEST_WEIGHTS))
print()
print('Kaydedilen model özeti:')
print(f'  n_qubits   = {int(data["n_qubits"])}')
print(f'  n_layers   = {int(data["n_layers"])}')
print(f'  feature_dim= {int(data["feature_dim"])}')
print(f'  output_dim = {data["quantum_weights"].shape[1] * 3}')
print(f'  n_classes  = {data["classifier_bias"].shape[0]}')